# Leave-One-Out Ablation (efficientnetv2 + maxvit)

**런타임: L4 GPU** 필수. 30 run(5 variant × 2 model × 3 fold), **단일 시드 1004**.

- 시드는 프로젝트 전체와 동일하게 **1004 고정 (변경 금지)**
- 표본 = 3-fold → 평균±std와 **3-fold 일관성**으로 보고
- 한 세션(약 2~3h)에 완료 가능. 셀 순서대로 실행.

In [ ]:
# 1) 클론 + 패키지
%cd /content
!rm -rf fireimage_detection
!git clone https://github.com/yuntaewon812/fireimage_detection.git fireimage_detection -q
!pip install timm grad-cam lime scikit-image scipy -q
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')

In [ ]:
# 2) 데이터 업로드 — fireimage_clean.zip (약 32MB) 하나만
#    (학습이므로 가중치 불필요 — 데이터만 있으면 됨)
from google.colab import files
up = files.upload()
print('업로드:', list(up.keys()))

In [ ]:
# 3) 데이터 압축 해제 (normal 792 / abnormal 727 확인)
%cd /content/fireimage_detection
!python colab_setup.py

In [ ]:
# 4) 학습 실행 — 단일 시드 1004 (5 variant × 2 model × 3 fold = 30 model-fold)
%cd /content/fireimage_detection
import time
t0 = time.time()
!python main_ablation_loo.py --seeds 1004 --epochs 15 --patience 5
print(f'\n총 학습 시간: {(time.time()-t0)/60:.1f}분')

In [ ]:
# 5) 집계 + 통계 (OOD-F1 mean±std, ΔOOD, GradCAM Sens/Stab)
%cd /content/fireimage_detection
!python aggregate_loo.py

In [ ]:
# 6) 결과 다운로드 (지표 CSV + 집계표, 작음)
import glob, os, zipfile
from google.colab import files
OUT = '/content/loo_results.zip'
with zipfile.ZipFile(OUT, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in glob.glob('results/fireimage_loo_*'):
        for f in glob.glob(os.path.join(p, '*.csv')):
            z.write(f, os.path.relpath(f, 'results'))
    if os.path.exists('results_LOO/loo_summary.csv'):
        z.write('results_LOO/loo_summary.csv', 'loo_summary.csv')
print('압축:', OUT)
files.download(OUT)

## (선택) 가중치 보관
재사용/추가분석용 가중치가 필요하면:
```python
import shutil
shutil.make_archive('/content/loo_weights', 'zip', 'model_save')
from google.colab import files; files.download('/content/loo_weights.zip')
```